In [24]:
import pandas as pd
import numpy as np

อันนี้คือขั้นก่อนเอาไฟล์เข้า

In [25]:
df = pd.read_csv("../data/raw/diabetes.csv")
df.shape

(768, 9)

ลองเอาแถวออกก่อน

In [26]:
df.info()
df.isna().sum()
df[['Glucose','BMI']].describe()
(df[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] == 0).sum()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

แถวซ้ำ

In [27]:
print(f"Shape of DataFrame before dropping duplicates: {df.shape}")
df.drop_duplicates(inplace=True)
print(f"Shape of DataFrame after dropping duplicates: {df.shape}")

Shape of DataFrame before dropping duplicates: (768, 9)
Shape of DataFrame after dropping duplicates: (768, 9)


In [28]:
df['Outcome_Label'] = df['Outcome'].map({0: 'Negative', 1: 'Positive'})
df['AgeGroup'] = pd.cut(df['Age'], bins=[20, 30, 40, 50, 60, 100],
    labels=['20s', '30s', '40s', '50s', '60+'], right=False)
df['Outcome'] = df['Outcome'].astype('category')

อันนี้ลองแก้สอดคล้อง


In [29]:
df['Outcome_Label'] = df['Outcome'].map({0: 'Negative', 1: 'Positive'})
df['AgeGroup'] = pd.cut(df['Age'], bins=[20, 30, 40, 50, 60, 100],
    labels=['20s', '30s', '40s', '50s', '60+'], right=False)
df['Outcome'] = df['Outcome'].astype('category')

จัดการค่าโดด

In [30]:
df[df['Insulin'] > 600]
df['SkinThickness'].describe()

count    768.000000
mean      20.536458
std       15.952218
min        0.000000
25%        0.000000
50%       23.000000
75%       32.000000
max       99.000000
Name: SkinThickness, dtype: float64

จัดการค่าหาย

In [31]:
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'BMI']
df[cols] = df[cols].replace(0, np.nan)
df[cols] = df.groupby('Outcome')[cols]\
    .transform(lambda x: x.fillna(x.median()))

df['Insulin'] = df['Insulin'].replace(0, np.nan)
df['HasInsulinReading'] = df['Insulin'].notna().astype(int)
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Outcome_Label,AgeGroup,HasInsulinReading
0,6,148.0,72.0,35.0,NaN,33.6,0.627,50,1,Positive,50s,0
1,1,85.0,66.0,29.0,NaN,26.6,0.351,31,0,Negative,30s,0
2,8,183.0,64.0,32.0,NaN,23.3,0.672,32,1,Positive,30s,0
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0,Negative,20s,1
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1,Positive,30s,1
...,...,...,...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0,Negative,60+,1
764,2,122.0,70.0,27.0,NaN,36.8,0.340,27,0,Negative,20s,0
765,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0,Negative,30s,1
766,1,126.0,60.0,32.0,NaN,30.1,0.349,47,1,Positive,40s,0


missing value

In [32]:
df_clean = df. dropna()
df_clean

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Outcome_Label,AgeGroup,HasInsulinReading
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0,Negative,20s,1
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1,Positive,30s,1
6,3,78.0,50.0,32.0,88.0,31.0,0.248,26,1,Positive,20s,1
8,2,197.0,70.0,45.0,543.0,30.5,0.158,53,1,Positive,50s,1
13,1,189.0,60.0,23.0,846.0,30.1,0.398,59,1,Positive,50s,1
...,...,...,...,...,...,...,...,...,...,...,...,...
753,0,181.0,88.0,44.0,510.0,43.3,0.222,26,1,Positive,20s,1
755,1,128.0,88.0,39.0,110.0,36.5,1.057,37,1,Positive,30s,1
760,2,88.0,58.0,26.0,16.0,28.4,0.766,22,0,Negative,20s,1
763,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0,Negative,60+,1


จัดการค่าโดด

In [33]:
def cap_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)
    return df

# Columns to apply outlier capping
outlier_columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

for col in outlier_columns:
    df = cap_outliers_iqr(df, col)
    print(f"Outliers capped for column: {col}")

print("\nDataFrame after outlier capping:")
display(df.head())

Outliers capped for column: Pregnancies
Outliers capped for column: Glucose
Outliers capped for column: BloodPressure
Outliers capped for column: SkinThickness
Outliers capped for column: Insulin
Outliers capped for column: BMI
Outliers capped for column: DiabetesPedigreeFunction
Outliers capped for column: Age

DataFrame after outlier capping:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Outcome_Label,AgeGroup,HasInsulinReading
0,6.0,148.0,72.0,35.0,NaN,33.6,0.627,50.0,1,Positive,50s,0
1,1.0,85.0,66.0,29.0,NaN,26.6,0.351,31.0,0,Negative,30s,0
2,8.0,183.0,64.0,32.0,NaN,23.3,0.672,32.0,1,Positive,30s,0
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0,0,Negative,20s,1
4,0.0,137.0,40.0,35.0,168.0,43.1,1.200,33.0,1,Positive,30s,1


In [34]:
print(f"Shape of DataFrame before dropping duplicates: {df.shape}")
df.drop_duplicates(inplace=True)
print(f"Shape of DataFrame after dropping duplicates: {df.shape}")
df

Shape of DataFrame before dropping duplicates: (768, 12)
Shape of DataFrame after dropping duplicates: (768, 12)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Outcome_Label,AgeGroup,HasInsulinReading
0,6.0,148.0,72.0,35.0,NaN,33.6,0.627,50.0,1,Positive,50s,0
1,1.0,85.0,66.0,29.0,NaN,26.6,0.351,31.0,0,Negative,30s,0
2,8.0,183.0,64.0,32.0,NaN,23.3,0.672,32.0,1,Positive,30s,0
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0,0,Negative,20s,1
4,0.0,137.0,40.0,35.0,168.0,43.1,1.200,33.0,1,Positive,30s,1
...,...,...,...,...,...,...,...,...,...,...,...,...
763,10.0,101.0,76.0,42.5,180.0,32.9,0.171,63.0,0,Negative,60+,1
764,2.0,122.0,70.0,27.0,NaN,36.8,0.340,27.0,0,Negative,20s,0
765,5.0,121.0,72.0,23.0,112.0,26.2,0.245,30.0,0,Negative,30s,1
766,1.0,126.0,60.0,32.0,NaN,30.1,0.349,47.0,1,Positive,40s,0


ค่าที่เป็นไปไม่ได้

In [35]:
# เติมเต็มค่า NaN ในคอลัมน์ 'Insulin' ด้วยมัธยฐานโดยแบ่งตาม 'Outcome'
df['Insulin'] = df.groupby('Outcome')['Insulin'].transform(lambda x: x.fillna(x.median()))

print('จำนวนค่าที่หายไปในแต่ละคอลัมน์ (NaNs) หลังการจัดการค่า Insuline:')
display(df.isna().sum())

จำนวนค่าที่หายไปในแต่ละคอลัมน์ (NaNs) หลังการจัดการค่า Insuline:


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
Outcome_Label               0
AgeGroup                    0
HasInsulinReading           0
dtype: int64

In [36]:
import pandas as pd
import numpy as np

# 1. โหลดข้อมูลต้นฉบับใหม่เพื่อเปรียบเทียบ 'ก่อน' การทำความสะอาด
df_original = pd.read_csv("../data/raw/diabetes.csv")
print("=== Statistics BEFORE Cleaning ===")
print(df_original.describe())

# 2. ทำสำเนาเพื่อดำเนินการทำความสะอาด
df_cleaned = df_original.copy()

# 3. ขั้นตอนการทำความสะอาดทั้งหมด (ตามที่เราได้ทำไปแล้ว)

# 3.1 จัดการค่า 0 ให้เป็น NaN ในคอลัมน์ที่เหมาะสม
# ตรวจสอบว่า `Outcome` เป็นประเภทที่ถูกต้องก่อนgroupby
df_cleaned['Outcome'] = df_cleaned['Outcome'].astype('category')
cols_to_replace_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_cleaned[cols_to_replace_zero] = df_cleaned[cols_to_replace_zero].replace(0, np.nan)

# 3.2 เติมค่า NaN ด้วยมัธยฐาน โดยอ้างอิงตาม 'Outcome'
for col in ['Glucose', 'BloodPressure', 'SkinThickness', 'BMI']:
    df_cleaned[col] = df_cleaned.groupby('Outcome')[col].transform(lambda x: x.fillna(x.median()))

# จัดการ Insulin แยกเนื่องจากมีการสร้าง HasInsulinReading
df_cleaned['HasInsulinReading'] = df_cleaned['Insulin'].notna().astype(int) # สร้างคอลัมน์นี้เผื่อไว้
df_cleaned['Insulin'] = df_cleaned.groupby('Outcome')['Insulin'].transform(lambda x: x.fillna(x.median()))

# 3.3 กำหนดฟังก์ชันสำหรับจัดการค่าโดด (Outlier Capping) ด้วย IQR
def cap_outliers_iqr(df_temp, column):
    Q1 = df_temp[column].quantile(0.25)
    Q3 = df_temp[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_temp[column] = df_temp[column].clip(lower=lower_bound, upper=upper_bound)
    return df_temp

# คอลัมน์ที่จะใช้จัดการค่าโดด
outlier_columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

# ใช้ฟังก์ชันจัดการค่าโดดกับคอลัมน์ที่กำหนด
for col in outlier_columns:
    df_cleaned = cap_outliers_iqr(df_cleaned, col)

# 3.4 จัดการแถวซ้ำ (ถึงแม้ในกรณีนี้จะไม่มี แต่เป็นขั้นตอนมาตรฐาน)
df_cleaned.drop_duplicates(inplace=True)

# 3.5 Feature Engineering (ตามที่ได้ทำไปแล้ว)
df_cleaned['Outcome_Label'] = df_cleaned['Outcome'].map({0: 'Negative', 1: 'Positive'})
df_cleaned['AgeGroup'] = pd.cut(df_cleaned['Age'], bins=[20, 30, 40, 50, 60, 100],
    labels=['20s', '30s', '40s', '50s', '60+'], right=False)


print("\n=== Statistics AFTER Cleaning ===")
print(df_cleaned.describe())

=== Statistics BEFORE Cleaning ===
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    0.476951  


In [38]:
df_cleaned.to_csv('../data/clean/diabetes_cleaned.csv', index=False)
print("ข้อมูลที่ทำความสะอาดแล้วถูกบันทึกเป็น 'diabetes_cleaned.csv'")

ข้อมูลที่ทำความสะอาดแล้วถูกบันทึกเป็น 'diabetes_cleaned.csv'
